# CPV VisDrone Training

**Phase 5 sanity:** `MODEL = 'yolov8n'`, `EPOCHS = 5`  
**Phase 6 full:** `EPOCHS = 50`, change `MODEL` per run (`yolov8n` / `yolov8m` / `rtdetr`)

Input: `itsdreowo/cpv-visdrone5`. No internet required.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL     = "yolov8n"                  # yolov8m | rtdetr  for Phase 6
EPOCHS    = 5                          # 5 = sanity, 50 = full
DATA_ROOT = "/kaggle/input/cpv-visdrone5"
DEVICE    = "cuda"
IMGSZ     = 640
SEED      = 42

WEIGHTS = {
    "yolov8n": "yolov8n.pt",
    "yolov8m": "yolov8m.pt",
    "rtdetr":  "rtdetr-l.pt",
}
BATCH = {"yolov8n": 16, "yolov8m": 16, "rtdetr": 8}

assert MODEL in WEIGHTS, f"MODEL must be one of {list(WEIGHTS)}"

In [ ]:
import os, pathlib, yaml

# Write dataset config inline — no external file needed
pathlib.Path("configs").mkdir(exist_ok=True)

data_cfg = {
    "path": DATA_ROOT,
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc": 5,
    "names": {0: "vehicle", 1: "person", 2: "static", 3: "flying", 4: "other"},
}
with open("configs/visdrone5.yaml", "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("Dataset config written.")
print(f"Training: model={MODEL}, epochs={EPOCHS}, batch={BATCH[MODEL]}, device={DEVICE}")

In [ ]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
from ultralytics import YOLO

model = YOLO(WEIGHTS[MODEL])
model.train(
    data="configs/visdrone5.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH[MODEL],
    device=DEVICE,
    seed=SEED,
    project="runs",
    name=MODEL,
)

In [ ]:
# Show training curves
import glob
from IPython.display import Image, display

for img in sorted(glob.glob("runs/*/results.png")):
    print(img)
    display(Image(img))

In [ ]:
# Copy best weights to notebook output for download
import shutil

for pt in sorted(pathlib.Path("runs").glob("*/weights/best.pt")):
    dest = pathlib.Path("/kaggle/working") / pt.parent.parent.name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, dest)
    print("Saved:", dest)